In [45]:
import pandas as pd

df_telemetry = pd.read_csv('aviation_data/fact_telemetry_high_freq.csv')
df_summary = pd.read_csv('aviation_data/flight_summary.csv')
df_registry = pd.read_csv('aviation_data/dim_aircraft.csv')

# 2 ETL and Cleaning
# 2.1 Make Relational Data - Merge 3 tables

master_df = pd.merge(df_telemetry, df_summary, on='flight_id', how='left')
master_df = pd.merge(master_df, df_registry, on='tail_number', how='left')

master_df.head()

,flight_id,timestamp_sec,altitude_ft,battery_soc_pct,phase,tail_number,payload_lbs,ambient_temp_c,ticket_revenue,model,battery_capacity_kwh,max_payload_lbs,firmware_version
0,FL-00001,0,0.0,99.97,CLIMB,N109AL,963.911348,38.328565,5121.87371,Alice_V1,850,2500,v2.2
1,FL-00001,1,33.3,99.94,CLIMB,N109AL,963.911348,38.328565,5121.87371,Alice_V1,850,2500,v2.2
2,FL-00001,2,66.6,99.92,CLIMB,N109AL,963.911348,38.328565,5121.87371,Alice_V1,850,2500,v2.2
3,FL-00001,3,99.9,99.89,CLIMB,N109AL,963.911348,38.328565,5121.87371,Alice_V1,850,2500,v2.2
4,FL-00001,4,133.2,99.86,CLIMB,N109AL,963.911348,38.328565,5121.87371,Alice_V1,850,2500,v2.2


In [ ]:
# 2.2 Cleaning & Tagging    
master_df['battery_soc_pct'] = master_df['battery_soc_pct'].ffill()

master_df['soc_raw'] = master_df['battery_soc_pct']

# Battery can not have negative energy
master_df['battery_soc_pct'] = master_df['battery_soc_pct'].clip(0, 100)
master_df['altitude_ft'] = master_df['altitude_ft'].clip(lower=0)

# Flag for FAA compliance (20%)
master_df['is_alert'] = master_df['soc_raw'] < 20.0

In [60]:
# 2.2 Summary & Preview
print(f"Total Telemetry records processed: {len(master_df)}")
print(f"Number of seconds in (SoC <= 0): {len(master_df[master_df['soc_raw'] <= 0])}")
print(f"Total 'is_alert' unique flights: {master_df[master_df['is_alert'] == True]['flight_id'].nunique()}")
master_df[['flight_id', 'timestamp_sec', 'soc_raw', 'is_alert']].tail(5)


Total Telemetry records processed: 450000
Number of seconds in (SoC <= 0): 7798
Total 'is_alert' unique flights: 100


,flight_id,timestamp_sec,soc_raw,is_alert
449995,FL-00100,4495,8.92,True
449996,FL-00100,4496,8.91,True
449997,FL-00100,4497,8.90,True
449998,FL-00100,4498,8.89,True
449999,FL-00100,4499,8.88,True


In [ ]:
# 2.3 Downsampling to -minute
master_df['timestamp_min'] = (master_df['timestamp_sec'] // 60) 

# Group by Flight and Minute, taking the Mean
agg_strategy = {
    'battery_soc_pct': 'mean',
    'soc_raw': 'mean',
    'altitude_ft': 'mean',
    'payload_lbs': 'first',
    'ambient_temp_c': 'first',
    'ticket_revenue': 'first',
    'firmware_version': 'first',
    'is_alert': 'max' 
}

analytical_view = master_df.groupby(['flight_id', 'timestamp_min']).agg(agg_strategy).reset_index()

analytical_view.to_csv('aviation_data/Alice_Master_Analysis_View.csv', index=False)

In [59]:
# 2.3 Summary
print(f"Downsampling: {len(analytical_view)} rows")
print(f"Ratio: {((1 - len(analytical_view)/len(master_df)) * 100):.1f}%")
analytical_view.head(10)

Downsampling: 7500 rows
Ratio: 98.3%


,flight_id,timestamp_min,battery_soc_pct,soc_raw,altitude_ft,payload_lbs,ambient_temp_c,ticket_revenue,firmware_version,is_alert
0,FL-00001,0,99.138000,99.138000,982.350,963.911348,38.328565,5121.87371,v2.2,False
1,FL-00001,1,97.442500,97.442500,2980.350,963.911348,38.328565,5121.87371,v2.2,False
2,FL-00001,2,95.746667,95.746667,4978.350,963.911348,38.328565,5121.87371,v2.2,False
3,FL-00001,3,94.051167,94.051167,6976.350,963.911348,38.328565,5121.87371,v2.2,False
4,FL-00001,4,92.355333,92.355333,8974.350,963.911348,38.328565,5121.87371,v2.2,False
5,FL-00001,5,90.659833,90.659833,10972.350,963.911348,38.328565,5121.87371,v2.2,False
6,FL-00001,6,88.964167,88.964167,12970.350,963.911348,38.328565,5121.87371,v2.2,False
7,FL-00001,7,87.322833,87.322833,14734.425,963.911348,38.328565,5121.87371,v2.2,False
8,FL-00001,8,85.996167,85.996167,15000.000,963.911348,38.328565,5121.87371,v2.2,False
9,FL-00001,9,84.721000,84.721000,15000.000,963.911348,38.328565,5121.87371,v2.2,False
